# Exploratory Data Analysis: WM-811K Wafer Defect Dataset

This notebook performs exploratory data analysis on the WM-811K wafer map dataset to understand:
- Dataset structure and dimensions
- Class distribution of defect types
- Sample wafer map visualizations
- Feature engineering exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Load Dataset

In [ ]:
DATA_PATH = '../data/LSWMD.pkl'
df = pd.read_pickle(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 2. Dataset Information

In [ ]:
print('=== Dataset Info ===')
print(f'Total samples: {len(df):,}')
print(f'Columns: {df.columns.tolist()}')
print()
df.info()

In [ ]:
# Check missing values
print('=== Missing Values ===')
print(df.isnull().sum())

## 3. Clean Dataset

In [ ]:
# Flatten failureType labels
def flatten_label(label):
    if isinstance(label, (list, np.ndarray)):
        if len(label) > 0:
            if isinstance(label[0], (list, np.ndarray)):
                return str(label[0][0]).strip()
            return str(label[0]).strip()
    return str(label).strip()

def is_valid_label(label):
    if label is None:
        return False
    if isinstance(label, (list, np.ndarray)):
        return len(label) > 0
    if isinstance(label, str):
        return len(label.strip()) > 0
    return True

# Apply cleaning
df = df.dropna(subset=['failureType'])
mask = df['failureType'].apply(is_valid_label)
df = df[mask].copy()
df['failureType'] = df['failureType'].apply(flatten_label)
df = df[df['failureType'].str.strip() != '']
df = df[df['failureType'].str.lower() != 'unknown']
df = df.reset_index(drop=True)

print(f'Cleaned dataset shape: {df.shape}')
print(f'Unique defect types: {df["failureType"].nunique()}')

## 4. Class Distribution Analysis

In [ ]:
# Class distribution
class_counts = df['failureType'].value_counts()
print('=== Class Distribution ===')
print(class_counts)
print()
print('=== Class Proportions ===')
print((class_counts / len(df) * 100).round(2).astype(str) + '%')

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
colors = sns.color_palette('husl', len(class_counts))
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors)
axes[0].set_title('Class Distribution (Bar Plot)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Defect Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Add count labels
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{count:,}', ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
           colors=colors, startangle=90)
axes[1].set_title('Class Distribution (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Sample Wafer Map Visualizations

In [ ]:
# Visualize sample wafer maps for each defect type
defect_types = df['failureType'].unique()
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, defect_type in enumerate(sorted(defect_types)):
    if i >= 9:
        break
    # Get a random sample of this defect type
    samples = df[df['failureType'] == defect_type]['waferMap'].values
    sample = samples[np.random.randint(len(samples))]
    
    axes[i].imshow(sample, cmap='RdYlGn_r', interpolation='nearest')
    axes[i].set_title(f'{defect_type}\n(Count: {class_counts[defect_type]:,})',
                     fontsize=12, fontweight='bold')
    axes[i].axis('off')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Wafer Maps by Defect Type', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Wafer Map Size Analysis

In [ ]:
# Analyze wafer map dimensions
heights = []
widths = []
for wm in df['waferMap'].values:
    if hasattr(wm, 'shape'):
        heights.append(wm.shape[0])
        widths.append(wm.shape[1])

print(f'Height range: {min(heights)} - {max(heights)}')
print(f'Width range: {min(widths)} - {max(widths)}')
print(f'Unique heights: {np.unique(heights)}')
print(f'Unique widths: {np.unique(widths)}')

## 7. Defect Ratio Analysis

In [ ]:
# Calculate failure ratio for each sample
failure_ratios = []
for wm in df['waferMap'].values:
    if hasattr(wm, 'shape') and wm.size > 0:
        ratio = np.sum(wm == 1) / wm.size
        failure_ratios.append(ratio)
    else:
        failure_ratios.append(0)

df['failure_ratio'] = failure_ratios

# Plot failure ratio distribution by class
fig, ax = plt.subplots(figsize=(14, 6))
for defect_type in sorted(defect_types):
    ratios = df[df['failureType'] == defect_type]['failure_ratio']
    ax.hist(ratios, bins=50, alpha=0.5, label=defect_type, density=True)

ax.set_xlabel('Failure Ratio', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Failure Ratio Distribution by Defect Type', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
# Summary statistics by class
print('=== Failure Ratio Statistics by Defect Type ===')
print(df.groupby('failureType')['failure_ratio'].describe().round(4))

## 9. Key Observations

### Dataset Characteristics:
1. **Class Imbalance**: The dataset shows significant class imbalance, with 'None' (no defect) being the majority class
2. **Defect Patterns**: Different defect types show distinct spatial patterns in their wafer maps
3. **Failure Ratios**: Defect types can be partially distinguished by their failure ratio distributions

### Implications for Modeling:
1. **Class balancing** techniques (class_weight, SMOTE, etc.) are necessary
2. **Spatial features** are critical for distinguishing between geometrically distinct defect patterns
3. **Statistical features** can capture distributional differences between defect types

### Next Steps:
1. Extract comprehensive features from wafer maps
2. Train Random Forest classifier with balanced class weights
3. Perform hyperparameter tuning
4. Evaluate and compare different approaches

In [ ]:
print('EDA Complete!')
print(f'Final cleaned dataset: {len(df):,} samples')
print(f'Number of classes: {df["failureType"].nunique()}')